In [1]:
import pandas as pd
import numpy as np

# === Datei laden ===
instances = [
    "a3_o80_m10_an10_ar9_reduced",
    "a5_o96_m10_an10_ar10_reduced",
    "a10_o107_m5_an57_ar12",
    "a10_o114_m6_an57_ar11",
    "a10_o128_m6_an51_ar13",
    "a10_o144_m6_an53_ar12",
    "a15_o170_m9_an80_ar18",
    "a20_o236_m12_an106_ar24",
    "a25_o306_m13_an127_ar31",
    "a30_o355_m18_an148_ar42",
    "a40_o476_m22_an215_ar51",
    "a50_o578_m28_an276_ar66"]

for instance in instances:
    print(f"Processing instance: {instance}")
    df = pd.read_csv(f"{instance}/TPSA/ParetoFront.csv")  # ggf. Pfad anpassen

    # === Zielspalten definieren ===
    all_objectives = [
        "Driver Violation",
        "Commute Distance",
        "Transport Machines",
        "Transport Attachments",
        "Machines",
        "Workers",
        "Attachments"
    ]

    # === 1. Paretofront aus Transport Attachments & Attachments extrahieren ===
    def pareto_front_2d(points):
        points = np.array(points)
        is_efficient = np.ones(points.shape[0], dtype=bool)
        for i, c in enumerate(points):
            if is_efficient[i]:
                is_efficient[is_efficient] = (
                    np.any(points[is_efficient] < c, axis=1)
                    | np.all(points[is_efficient] == c, axis=1)
                )
                is_efficient[i] = True
        return points[is_efficient]

    pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
    df_pareto_attach = pd.DataFrame(
        pareto_points, columns=["Transport Attachments", "Attachments"]
    ).drop_duplicates()

    # === 2. Lösungen erweitern ===
    df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(
        df_pareto_attach, how="cross"
    )

    # === 3. Finaler Paretofilter (alle Ziele) ===
    def pareto_filter_nd(df, objective_cols):
        values = df[objective_cols].values
        is_efficient = np.ones(values.shape[0], dtype=bool)
        for i, v in enumerate(values):
            if is_efficient[i]:
                is_efficient[is_efficient] = (
                    np.any(values[is_efficient] < v, axis=1)
                    | np.all(values[is_efficient] == v, axis=1)
                )
                is_efficient[i] = True
        return df[is_efficient].reset_index(drop=True)

    df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

    # === Insights ===
    print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
    print(f"📦 Ursprüngliche Lösungen:         {len(df)}")
    print(f"🎯 Pareto-Kombinationen (2D):      {len(df_pareto_attach)}")
    print(f"🧩 Erweiterte Lösungskombis:       {len(df_expanded)}")
    print(f"✅ Nicht-dominierte Endlösungen:   {len(df_final_pareto)}")
    print(f"❌ Entfernte (dominierte) Lösungen: {len(df_expanded) - len(df_final_pareto)}\n")

    print("📊 Pareto-Kombinationen (Anbaugeräte):")
    print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

    # === Ergebnis speichern ===
    df_final_pareto.to_csv(f"{instance}/TPSA/ParetoFront_filtered.csv", index=False)
    print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")

Processing instance: a3_o80_m10_an10_ar9_reduced
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         108
🎯 Pareto-Kombinationen (2D):      1
🧩 Erweiterte Lösungskombis:       108
✅ Nicht-dominierte Endlösungen:   84
❌ Entfernte (dominierte) Lösungen: 24

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
                   0.0          2.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv
Processing instance: a5_o96_m10_an10_ar10_reduced
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         127
🎯 Pareto-Kombinationen (2D):      1
🧩 Erweiterte Lösungskombis:       127
✅ Nicht-dominierte Endlösungen:   127
❌ Entfernte (dominierte) Lösungen: 0

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
                 29.73          4.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv
Processing instance: a10_o107_m5_an57_ar12
🔍 Insights zur P